# Establecimientos educacionales y clientes regulados probables

Este notebook estima, por comuna, la cantidad y proporcion de establecimientos educacionales escolares publicos y comerciales que no aparecen en la base mensual de ventas a clientes libres electricos.

La clasificacion se basa en `COD_DEPE2` del Directorio Oficial de Establecimientos Educacionales 2025. El cruce con clientes libres es nominal, usando `NOM_RBD` contra `RazonSocialCliente`; por lo tanto, un establecimiento sin match se interpreta como cliente regulado probable, no como prueba administrativa definitiva.

## 1. Dependencias y rutas

In [1]:
import importlib.util
from pathlib import Path

REQUIRED_PACKAGES = ["pandas"]
missing = [pkg for pkg in REQUIRED_PACKAGES if importlib.util.find_spec(pkg) is None]

if missing:
    raise ImportError(
        "Faltan dependencias para ejecutar este notebook: "
        + ", ".join(missing)
        + ". Instalar antes de continuar, por ejemplo con `pip install pandas`."
    )

print("Dependencias disponibles:", ", ".join(REQUIRED_PACKAGES))

Dependencias disponibles: pandas


In [2]:
import re
import unicodedata
from collections import defaultdict
from difflib import SequenceMatcher

import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "prototipo_2" / "data").exists() and (candidate / "prototipo_3").exists():
            return candidate
    raise FileNotFoundError("No se encontro la raiz MERLIN_EDM desde el directorio actual.")


ROOT = find_repo_root()
DATA_EDUCACION = ROOT / "prototipo_3" / "data" / "Directorio-Oficial-EE-2025" / "20250926_Directorio_Oficial_EE_2025_20250430_WEB.csv"
DATA_CLIENTES_LIBRES = ROOT / "prototipo_2" / "data" / "venta_clientes_libres.csv"
OUTPUT_DIR = ROOT / "prototipo_3" / "data" / "interim"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for path in [DATA_EDUCACION, DATA_CLIENTES_LIBRES, OUTPUT_DIR]:
    print(path, "OK" if path.exists() else "NO EXISTE")

c:\Users\Raimundo Claren\Documents\MERLIN_EDM\prototipo_3\data\Directorio-Oficial-EE-2025\20250926_Directorio_Oficial_EE_2025_20250430_WEB.csv OK
c:\Users\Raimundo Claren\Documents\MERLIN_EDM\prototipo_2\data\venta_clientes_libres.csv OK
c:\Users\Raimundo Claren\Documents\MERLIN_EDM\prototipo_3\data\interim OK


## 2. Reglas de clasificacion y normalizacion

In [3]:
DEPENDENCIA_MAP = {
    1: "Municipal",
    2: "Particular subvencionado",
    3: "Particular pagado",
    4: "Corporacion de administracion delegada",
    5: "Servicio local de educacion",
}

TIPO_DEPENDENCIA_MAP = {
    1: "publico",
    4: "publico",
    5: "publico",
    2: "comercial",
    3: "comercial",
}

LEGAL_TERMS = {
    "s", "sa", "saa", "spa", "ltda", "limitada", "soc", "sociedad", "anonima",
    "cia", "compania", "company", "inc", "corp", "corporacion", "fundacion",
    "inmobiliaria", "comercial", "servicios"
}

CONNECTOR_TERMS = {"de", "del", "la", "el", "los", "las", "y", "e", "en", "para", "por", "a"}

SCHOOL_GENERIC_TERMS = {
    "colegio", "escuela", "esc", "liceo", "l", "instituto", "centro", "educacional",
    "educacion", "educativa", "educativo", "parvulario", "jardin", "infantil", "basica",
    "media", "bicentenario", "politecnico", "tecnico", "profesional", "tp", "especial"
}

EDUCATION_SIGNAL_TERMS = {
    "colegio", "escuela", "esc", "liceo", "educacional", "educacion", "educativa",
    "educativo", "parvulario"
}

HIGHER_ED_TERMS = {"universidad", "duoc", "inacap", "instituto profesional", "ip"}


def strip_accents(text):
    text = "" if pd.isna(text) else str(text)
    return unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("ascii")


def tokenize(text):
    text = strip_accents(text).lower().replace("&", " y ")
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return [tok for tok in text.split() if tok]


def normalize_name(text, remove_school_terms=False):
    drop = LEGAL_TERMS | CONNECTOR_TERMS
    if remove_school_terms:
        drop = drop | SCHOOL_GENERIC_TERMS
    return " ".join(tok for tok in tokenize(text) if tok not in drop)


def aliases_establecimiento(nombre):
    base = str(nombre)
    variants = {
        normalize_name(base),
        normalize_name(base, remove_school_terms=True),
        normalize_name(re.sub(r"\besc\.?\b", "escuela", strip_accents(base), flags=re.IGNORECASE)),
        normalize_name(re.sub(r"\bl\.?\b", "liceo", strip_accents(base), flags=re.IGNORECASE)),
    }
    return sorted(v for v in variants if len(v) >= 4)


def aliases_cliente(nombre):
    base = str(nombre)
    variants = {normalize_name(base), normalize_name(base, remove_school_terms=True)}
    return sorted(v for v in variants if len(v) >= 4)


def sequence_score(a, b):
    if not a or not b:
        return 0.0
    return SequenceMatcher(None, a, b).ratio()


def token_score(a, b):
    toks_a = set(a.split())
    toks_b = set(b.split())
    if not toks_a or not toks_b:
        return 0.0
    overlap = len(toks_a & toks_b)
    dice = 2 * overlap / (len(toks_a) + len(toks_b))
    containment = overlap / min(len(toks_a), len(toks_b))
    ordered_a = " ".join(sorted(toks_a))
    ordered_b = " ".join(sorted(toks_b))
    return max(dice, containment * 0.96, sequence_score(ordered_a, ordered_b))


def similarity(a, b):
    return max(sequence_score(a, b), token_score(a, b))


def has_education_signal(text):
    norm = normalize_name(text)
    toks = set(norm.split())
    if toks & EDUCATION_SIGNAL_TERMS:
        return True
    return "jardin infantil" in norm


def has_higher_ed_signal(text):
    norm = normalize_name(text)
    toks = set(norm.split())
    return "universidad" in toks or "duoc" in toks or "inacap" in toks or "instituto profesional" in norm


def distinctive_tokens(alias):
    generic = LEGAL_TERMS | CONNECTOR_TERMS | SCHOOL_GENERIC_TERMS | {"san", "santa", "santo"}
    return [tok for tok in alias.split() if tok not in generic and len(tok) >= 4]


def comuna_consistente(comuna_establecimiento, comuna_cliente):
    comuna_e = normalize_name(comuna_establecimiento)
    comuna_c = normalize_name(comuna_cliente)
    if not comuna_e or not comuna_c:
        return False
    return comuna_e in comuna_c or comuna_c in comuna_e

## 3. Carga y diagnostico del Directorio Oficial EE 2025

In [4]:
df_ee_raw = pd.read_csv(DATA_EDUCACION, sep=";", encoding="utf-8-sig", dtype="string")
df_ee_raw.columns = [col.strip() for col in df_ee_raw.columns]

required_ee = ["AGNO", "RBD", "NOM_RBD", "COD_COM_RBD", "NOM_COM_RBD", "COD_DEPE2", "ESTADO_ESTAB", "MAT_TOTAL"]
missing_cols = [col for col in required_ee if col not in df_ee_raw.columns]
if missing_cols:
    raise ValueError(f"Columnas faltantes en directorio educacional: {missing_cols}")

for col in ["COD_DEPE2", "ESTADO_ESTAB", "MAT_TOTAL"]:
    df_ee_raw[col] = pd.to_numeric(df_ee_raw[col], errors="coerce")

print(f"Filas totales directorio EE: {len(df_ee_raw):,}")
print("Distribucion COD_DEPE2:")
display(df_ee_raw["COD_DEPE2"].value_counts(dropna=False).sort_index().rename_axis("COD_DEPE2").reset_index(name="n"))
print("Distribucion ESTADO_ESTAB:")
display(df_ee_raw["ESTADO_ESTAB"].value_counts(dropna=False).sort_index().rename_axis("ESTADO_ESTAB").reset_index(name="n"))

Filas totales directorio EE: 16,768
Distribucion COD_DEPE2:


,COD_DEPE2,n
0,1,4696
1,2,7658
2,3,2183
3,4,70
4,5,2161


Distribucion ESTADO_ESTAB:


,ESTADO_ESTAB,n
0,1,12038
1,2,782
2,3,3870
3,4,78


In [5]:
EXPECTED_DEP_COUNTS = {1: 4696, 2: 7658, 3: 2183, 4: 70, 5: 2161}
actual_dep_counts = df_ee_raw["COD_DEPE2"].value_counts().to_dict()
for code, expected in EXPECTED_DEP_COUNTS.items():
    actual = int(actual_dep_counts.get(code, 0))
    print(f"COD_DEPE2={code}: observado={actual:,} esperado={expected:,} -> {'OK' if actual == expected else 'REVISAR'}")

activos = int((df_ee_raw["ESTADO_ESTAB"] == 1).sum())
print(f"Establecimientos activos ESTADO_ESTAB=1: {activos:,} -> {'OK' if activos == 12038 else 'REVISAR'}")

COD_DEPE2=1: observado=4,696 esperado=4,696 -> OK
COD_DEPE2=2: observado=7,658 esperado=7,658 -> OK
COD_DEPE2=3: observado=2,183 esperado=2,183 -> OK
COD_DEPE2=4: observado=70 esperado=70 -> OK
COD_DEPE2=5: observado=2,161 esperado=2,161 -> OK
Establecimientos activos ESTADO_ESTAB=1: 12,038 -> OK


## 4. Preparacion de establecimientos educacionales activos

In [6]:
df_ee = df_ee_raw[df_ee_raw["ESTADO_ESTAB"].eq(1)].copy()

df_ee["glosa_dependencia"] = df_ee["COD_DEPE2"].map(DEPENDENCIA_MAP)
df_ee["tipo_dependencia"] = df_ee["COD_DEPE2"].map(TIPO_DEPENDENCIA_MAP)
df_ee["nombre_establecimiento"] = df_ee["NOM_RBD"].fillna("").str.strip()
df_ee["comuna"] = df_ee["NOM_COM_RBD"].fillna("").str.strip().str.upper()
df_ee["cod_comuna"] = df_ee["COD_COM_RBD"].fillna("").str.strip()
df_ee["matricula_total"] = pd.to_numeric(df_ee["MAT_TOTAL"], errors="coerce").fillna(0)
df_ee["establecimiento_norm"] = df_ee["nombre_establecimiento"].map(normalize_name)
df_ee["establecimiento_aliases"] = df_ee["nombre_establecimiento"].map(aliases_establecimiento)

if df_ee["tipo_dependencia"].isna().any():
    display(df_ee[df_ee["tipo_dependencia"].isna()][["RBD", "NOM_RBD", "COD_DEPE2"]].head(20))
    raise ValueError("Hay establecimientos activos con COD_DEPE2 no mapeado.")

print(f"Establecimientos activos preparados: {len(df_ee):,}")
display(df_ee[["RBD", "nombre_establecimiento", "cod_comuna", "comuna", "COD_DEPE2", "glosa_dependencia", "tipo_dependencia", "matricula_total"]].head(10))

Establecimientos activos preparados: 12,038


,RBD,nombre_establecimiento,cod_comuna,comuna,COD_DEPE2,glosa_dependencia,tipo_dependencia,matricula_total
0,1,LICEO POLITECNICO ARICA,15101,ARICA,5,Servicio local de educacion,publico,1086
1,2,PARVULARIO LAS ESPIGUITAS,15101,ARICA,5,Servicio local de educacion,publico,112
2,3,ESC. PEDRO VICENTE GUTIERREZ TORRES,15101,ARICA,5,Servicio local de educacion,publico,497
3,4,LICEO OCTAVIO PALMA PEREZ,15101,ARICA,5,Servicio local de educacion,publico,1279
4,5,JOVINA NARANJO FERNANDEZ,15101,ARICA,5,Servicio local de educacion,publico,783
5,7,L. POLI. BICENTENARIO DE EXCELENCIA ANTONIO VARAS DE LA BARRA,15101,ARICA,5,Servicio local de educacion,publico,1110
6,8,COLEGIO EDUARDO FREI MONTALVA,15101,ARICA,5,Servicio local de educacion,publico,806
7,9,ESCUELA REPUBLICA DE ISRAEL,15101,ARICA,5,Servicio local de educacion,publico,1379
8,10,ESCUELA REPUBLICA DE FRANCIA,15101,ARICA,5,Servicio local de educacion,publico,137
9,11,ESC. GRAL. PEDRO LAGOS MARCHANT,15101,ARICA,5,Servicio local de educacion,publico,322


## 5. Preparacion de clientes libres

In [7]:
df_ventas_raw = pd.read_csv(DATA_CLIENTES_LIBRES, encoding="utf-8-sig")
required_ventas = ["Anio", "NroMes", "RazonSocialCliente", "Comuna", "Region", "RetiroEnergiaMWh"]
missing_cols = [col for col in required_ventas if col not in df_ventas_raw.columns]
if missing_cols:
    raise ValueError(f"Columnas faltantes en ventas a clientes libres: {missing_cols}")

df_ventas = df_ventas_raw.copy()
df_ventas["RazonSocialCliente"] = df_ventas["RazonSocialCliente"].fillna("").astype(str).str.strip()
df_ventas["cliente_norm"] = df_ventas["RazonSocialCliente"].map(normalize_name)
df_ventas["RetiroEnergiaMWh"] = pd.to_numeric(df_ventas["RetiroEnergiaMWh"], errors="coerce")
df_ventas["Anio"] = pd.to_numeric(df_ventas["Anio"], errors="coerce").astype("Int64")

df_clientes = (
    df_ventas[df_ventas["cliente_norm"].ne("")]
    .groupby("cliente_norm", as_index=False)
    .agg(
        razon_social_cliente=("RazonSocialCliente", lambda s: sorted(set(s))[0]),
        variantes_razon_social=("RazonSocialCliente", lambda s: " | ".join(sorted(set(s))[:8])),
        comuna_cliente=("Comuna", lambda s: " | ".join(sorted(set(s.dropna().astype(str)))[:8])),
        region_cliente=("Region", lambda s: " | ".join(sorted(set(s.dropna().astype(str)))[:8])),
        anio_min=("Anio", "min"),
        anio_max=("Anio", "max"),
        n_meses_registrados=("RetiroEnergiaMWh", "count"),
        retiro_mwh_total=("RetiroEnergiaMWh", "sum"),
        retiro_mwh_promedio_mensual=("RetiroEnergiaMWh", "mean"),
    )
)
df_clientes["cliente_aliases"] = df_clientes["razon_social_cliente"].map(aliases_cliente)
df_clientes["tiene_senal_educacional"] = df_clientes["razon_social_cliente"].map(has_education_signal)
df_clientes["posible_educacion_superior"] = df_clientes["razon_social_cliente"].map(has_higher_ed_signal)

print(f"Filas ventas clientes libres: {len(df_ventas_raw):,}")
print(f"Razones sociales normalizadas unicas: {len(df_clientes):,}")
print(f"Razones sociales con senal educacional: {int(df_clientes['tiene_senal_educacional'].sum()):,}")
display(df_clientes[df_clientes["tiene_senal_educacional"]][["razon_social_cliente", "comuna_cliente", "region_cliente", "anio_min", "anio_max"]].sort_values("razon_social_cliente").head(50))

Filas ventas clientes libres: 311,887
Razones sociales normalizadas unicas: 2,127
Razones sociales con senal educacional: 3


,razon_social_cliente,comuna_cliente,region_cliente,anio_min,anio_max
649,colegio aleman pto. varas,PUERTO VARAS,LOS LAGOS,2018,2023
824,corporacion educacional emprender,OSORNO,LOS LAGOS,2018,2020
650,sociedad inmobiliaria colegio de ingenieros spa,RECOLETA,METROPOLITANA DE SANTIAGO,2018,2023


## 6. Indice de candidatos y cruce nominal

In [8]:
cliente_records = df_clientes.to_dict("records")
token_to_clientes = defaultdict(set)
education_client_idx = set()

for idx, row in enumerate(cliente_records):
    if row["tiene_senal_educacional"]:
        education_client_idx.add(idx)
    for alias in row["cliente_aliases"]:
        for tok in distinctive_tokens(alias):
            token_to_clientes[tok].add(idx)


def candidate_indices_for_establishment(estab_aliases):
    candidates = set()
    for alias in estab_aliases:
        for tok in distinctive_tokens(alias):
            candidates.update(token_to_clientes.get(tok, set()))
    # Siempre considerar clientes con senal educacional, porque son pocos y pueden usar nombres institucionales.
    candidates.update(education_client_idx)
    return candidates


def best_candidate_for_establishment(estab_row):
    best = None
    estab_aliases = estab_row["establecimiento_aliases"]
    candidate_indices = candidate_indices_for_establishment(estab_aliases)

    for idx in candidate_indices:
        cli = cliente_records[idx]
        local_best = {"score": 0.0, "regla_match": "sin_match", "alias_establecimiento": "", "alias_cliente": ""}
        for alias_e in estab_aliases:
            for alias_c in cli["cliente_aliases"]:
                if alias_e == alias_c:
                    score = 1.0
                    regla = "exacto_normalizado"
                elif alias_e in alias_c or alias_c in alias_e:
                    score = min(0.97, similarity(alias_e, alias_c))
                    regla = "contencion_alias"
                else:
                    score = similarity(alias_e, alias_c)
                    regla = "difuso"
                if score > local_best["score"]:
                    local_best = {
                        "score": score,
                        "regla_match": regla,
                        "alias_establecimiento": alias_e,
                        "alias_cliente": alias_c,
                    }

        if best is None or local_best["score"] > best["score"]:
            best = {**cli, **local_best, "n_candidatos_evaluados": len(candidate_indices)}

    if best is None:
        best = {
            "cliente_norm": "", "razon_social_cliente": "", "variantes_razon_social": "",
            "comuna_cliente": "", "region_cliente": "", "anio_min": pd.NA, "anio_max": pd.NA,
            "n_meses_registrados": 0, "retiro_mwh_total": 0.0, "retiro_mwh_promedio_mensual": pd.NA,
            "cliente_aliases": [], "tiene_senal_educacional": False, "posible_educacion_superior": False,
            "score": 0.0, "regla_match": "sin_match", "alias_establecimiento": "", "alias_cliente": "",
            "n_candidatos_evaluados": 0,
        }
    return best


records = []
for i, (_, estab) in enumerate(df_ee.iterrows(), start=1):
    if i == 1 or i % 1000 == 0 or i == len(df_ee):
        print(f"Procesando establecimiento {i:,}/{len(df_ee):,}")
    best = best_candidate_for_establishment(estab)
    records.append({**estab.to_dict(), **best})

df_cruce = pd.DataFrame(records)
print("Cruce nominal completado.")
display(df_cruce[["RBD", "nombre_establecimiento", "razon_social_cliente", "score", "regla_match", "tiene_senal_educacional"]].sort_values("score", ascending=False).head(30))

Procesando establecimiento 1/12,038
Procesando establecimiento 1,000/12,038
Procesando establecimiento 2,000/12,038
Procesando establecimiento 3,000/12,038
Procesando establecimiento 4,000/12,038
Procesando establecimiento 5,000/12,038
Procesando establecimiento 6,000/12,038
Procesando establecimiento 7,000/12,038
Procesando establecimiento 8,000/12,038
Procesando establecimiento 9,000/12,038
Procesando establecimiento 10,000/12,038
Procesando establecimiento 11,000/12,038
Procesando establecimiento 12,000/12,038
Procesando establecimiento 12,038/12,038
Cruce nominal completado.


,RBD,nombre_establecimiento,razon_social_cliente,score,regla_match,tiene_senal_educacional
3788,6118,COLEGIO SANTA CRUZ,santa cruz s.a.,1.0,exacto_normalizado,False
279,413,ESCUELA LAS BRISAS,inmobiliaria las brisas s.a.,1.0,exacto_normalizado,False
11445,40459,CENTRO EDUCACIONAL ALBORADA,servicios alborada spa,1.0,exacto_normalizado,False
3158,5010,ESCUELA METODISTA,corporacion metodista,1.0,exacto_normalizado,False
4181,6895,LICEO BICENTENARIO SANTA CRUZ,santa cruz s.a.,1.0,exacto_normalizado,False
7747,15705,COLEGIO SANTA TERESA,santa teresa s.a.,1.0,exacto_normalizado,False
3885,6304,ESCUELA ALBORADA,servicios alborada spa,1.0,exacto_normalizado,False
3123,4961,ESCUELA BASICA COLCURA,colcura s.a.,1.0,exacto_normalizado,False
3637,5814,COLEGIO SANTA CRUZ,santa cruz s.a.,1.0,exacto_normalizado,False
6628,11950,COLEGIO MONTEVERDE,monteverde s. a.,1.0,exacto_normalizado,False


## 7. Clasificacion de cliente libre vs regulado probable

In [9]:
df_cruce["comuna_consistente"] = df_cruce.apply(lambda r: comuna_consistente(r["comuna"], r["comuna_cliente"]), axis=1)


def classify_match(row):
    score = row["score"]
    has_signal = bool(row["tiene_senal_educacional"])
    higher_ed = bool(row["posible_educacion_superior"])
    regla = row["regla_match"]
    comuna_ok = bool(row["comuna_consistente"])

    if not has_signal:
        if score >= 0.88:
            return "revision_manual"
        return "sin_match_cliente_libre"

    if higher_ed and score < 0.96:
        return "revision_manual"

    if regla == "exacto_normalizado" and score >= 0.98:
        return "cliente_libre_alta_confianza"
    if score >= 0.93 and comuna_ok:
        return "cliente_libre_alta_confianza"
    if score >= 0.84 and comuna_ok:
        return "cliente_libre_probable"
    if score >= 0.74:
        return "revision_manual"
    return "sin_match_cliente_libre"


df_cruce["clasificacion"] = df_cruce.apply(classify_match, axis=1)
df_cruce["cliente_libre_defendible"] = df_cruce["clasificacion"].isin(["cliente_libre_alta_confianza", "cliente_libre_probable"])
df_cruce["cliente_regulado_probable"] = ~df_cruce["cliente_libre_defendible"]
df_cruce["requiere_revision_manual"] = df_cruce["clasificacion"].eq("revision_manual")
df_cruce["nota_metodologica"] = df_cruce.apply(
    lambda r: "posible educacion superior/no escolar" if r["posible_educacion_superior"] else (
        "candidato sin senal educacional explicita" if not r["tiene_senal_educacional"] and r["score"] >= 0.74 else ""
    ),
    axis=1,
)

resumen_clasificacion = (
    df_cruce["clasificacion"].value_counts().rename_axis("clasificacion").reset_index(name="n_establecimientos")
)
resumen_clasificacion["porcentaje"] = resumen_clasificacion["n_establecimientos"] / len(df_cruce) * 100

display(resumen_clasificacion)
display(df_cruce[["clasificacion", "RBD", "nombre_establecimiento", "comuna", "tipo_dependencia", "razon_social_cliente", "score", "regla_match", "nota_metodologica"]].sort_values("score", ascending=False).head(40))

,clasificacion,n_establecimientos,porcentaje
0,sin_match_cliente_libre,11147,92.598438
1,revision_manual,889,7.384948
2,cliente_libre_alta_confianza,2,0.016614


,clasificacion,RBD,nombre_establecimiento,comuna,tipo_dependencia,razon_social_cliente,score,regla_match,nota_metodologica
3788,revision_manual,6118,COLEGIO SANTA CRUZ,VILLARRICA,comercial,santa cruz s.a.,1.00,exacto_normalizado,candidato sin senal educacional explicita
279,revision_manual,413,ESCUELA LAS BRISAS,COPIAPÓ,publico,inmobiliaria las brisas s.a.,1.00,exacto_normalizado,candidato sin senal educacional explicita
11445,revision_manual,40459,CENTRO EDUCACIONAL ALBORADA,ANCUD,comercial,servicios alborada spa,1.00,exacto_normalizado,candidato sin senal educacional explicita
3158,revision_manual,5010,ESCUELA METODISTA,CORONEL,comercial,corporacion metodista,1.00,exacto_normalizado,candidato sin senal educacional explicita
4181,revision_manual,6895,LICEO BICENTENARIO SANTA CRUZ,MARIQUINA,comercial,santa cruz s.a.,1.00,exacto_normalizado,candidato sin senal educacional explicita
7747,revision_manual,15705,COLEGIO SANTA TERESA,RANCAGUA,comercial,santa teresa s.a.,1.00,exacto_normalizado,candidato sin senal educacional explicita
3885,revision_manual,6304,ESCUELA ALBORADA,LONCOCHE,publico,servicios alborada spa,1.00,exacto_normalizado,candidato sin senal educacional explicita
3123,revision_manual,4961,ESCUELA BASICA COLCURA,LOTA,publico,colcura s.a.,1.00,exacto_normalizado,candidato sin senal educacional explicita
3637,revision_manual,5814,COLEGIO SANTA CRUZ,TEMUCO,comercial,santa cruz s.a.,1.00,exacto_normalizado,candidato sin senal educacional explicita
6628,revision_manual,11950,COLEGIO MONTEVERDE,PEÑALOLÉN,comercial,monteverde s. a.,1.00,exacto_normalizado,candidato sin senal educacional explicita


## 8. Resumen comunal de regulados probables

In [10]:
df_cruce["n_publico"] = (df_cruce["tipo_dependencia"] == "publico").astype(int)
df_cruce["n_comercial"] = (df_cruce["tipo_dependencia"] == "comercial").astype(int)
df_cruce["n_publico_libre"] = ((df_cruce["tipo_dependencia"] == "publico") & df_cruce["cliente_libre_defendible"]).astype(int)
df_cruce["n_comercial_libre"] = ((df_cruce["tipo_dependencia"] == "comercial") & df_cruce["cliente_libre_defendible"]).astype(int)
df_cruce["n_publico_regulado_probable"] = ((df_cruce["tipo_dependencia"] == "publico") & df_cruce["cliente_regulado_probable"]).astype(int)
df_cruce["n_comercial_regulado_probable"] = ((df_cruce["tipo_dependencia"] == "comercial") & df_cruce["cliente_regulado_probable"]).astype(int)
df_cruce["matricula_publica"] = df_cruce["matricula_total"].where(df_cruce["tipo_dependencia"] == "publico", 0)
df_cruce["matricula_comercial"] = df_cruce["matricula_total"].where(df_cruce["tipo_dependencia"] == "comercial", 0)
df_cruce["matricula_publica_regulada_probable"] = df_cruce["matricula_total"].where((df_cruce["tipo_dependencia"] == "publico") & df_cruce["cliente_regulado_probable"], 0)
df_cruce["matricula_comercial_regulada_probable"] = df_cruce["matricula_total"].where((df_cruce["tipo_dependencia"] == "comercial") & df_cruce["cliente_regulado_probable"], 0)

resumen_comunal = (
    df_cruce.groupby(["cod_comuna", "comuna"], as_index=False)
    .agg(
        total_establecimientos=("RBD", "count"),
        total_publicos=("n_publico", "sum"),
        total_comerciales=("n_comercial", "sum"),
        publicos_cliente_libre=("n_publico_libre", "sum"),
        comerciales_cliente_libre=("n_comercial_libre", "sum"),
        publicos_regulados_probables=("n_publico_regulado_probable", "sum"),
        comerciales_regulados_probables=("n_comercial_regulado_probable", "sum"),
        matricula_total=("matricula_total", "sum"),
        matricula_publica=("matricula_publica", "sum"),
        matricula_comercial=("matricula_comercial", "sum"),
        matricula_publica_regulada_probable=("matricula_publica_regulada_probable", "sum"),
        matricula_comercial_regulada_probable=("matricula_comercial_regulada_probable", "sum"),
    )
)

resumen_comunal["prop_publica_regulada_probable"] = resumen_comunal["publicos_regulados_probables"] / resumen_comunal["total_publicos"].replace({0: pd.NA})
resumen_comunal["prop_comercial_regulada_probable"] = resumen_comunal["comerciales_regulados_probables"] / resumen_comunal["total_comerciales"].replace({0: pd.NA})
resumen_comunal["prop_matricula_publica_regulada_probable"] = resumen_comunal["matricula_publica_regulada_probable"] / resumen_comunal["matricula_publica"].replace({0: pd.NA})
resumen_comunal["prop_matricula_comercial_regulada_probable"] = resumen_comunal["matricula_comercial_regulada_probable"] / resumen_comunal["matricula_comercial"].replace({0: pd.NA})

display(resumen_comunal.sort_values(["total_establecimientos", "comuna"], ascending=[False, True]).head(30))

,cod_comuna,comuna,total_establecimientos,total_publicos,total_comerciales,publicos_cliente_libre,comerciales_cliente_libre,publicos_regulados_probables,comerciales_regulados_probables,matricula_total,matricula_publica,matricula_comercial,matricula_publica_regulada_probable,matricula_comercial_regulada_probable,prop_publica_regulada_probable,prop_comercial_regulada_probable,prop_matricula_publica_regulada_probable,prop_matricula_comercial_regulada_probable
85,13201,PUENTE ALTO,213,28,185,0,0,28,185,101566,14599,86967,14599,86967,1.0,1.0,1.0,1.0
71,13119,MAIPÚ,210,28,182,0,0,28,182,90386,15934,74452,15934,74452,1.0,1.0,1.0,1.0
62,13110,LA FLORIDA,204,26,178,0,0,26,178,68708,10430,58278,10430,58278,1.0,1.0,1.0,1.0
186,5109,VIÑA DEL MAR,204,50,154,0,0,50,154,57118,12754,44364,12754,44364,1.0,1.0,1.0,1.0
165,4101,LA SERENA,200,42,158,0,0,42,158,59967,13464,46503,13464,46503,1.0,1.0,1.0,1.0
0,10101,PUERTO MONTT,183,76,107,0,0,76,107,56719,19651,37068,19651,37068,1.0,1.0,1.0,1.0
314,9101,TEMUCO,179,44,135,0,0,44,135,64615,15845,48770,15845,48770,1.0,1.0,1.0,1.0
180,5101,VALPARAÍSO,175,55,120,0,0,55,120,50370,15992,34378,15992,34378,1.0,1.0,1.0,1.0
218,6101,RANCAGUA,157,39,118,0,0,39,118,53664,20365,33299,20365,33299,1.0,1.0,1.0,1.0
166,4102,COQUIMBO,154,39,115,0,0,39,115,48020,12879,35141,12879,35141,1.0,1.0,1.0,1.0


## 9. Exportacion de resultados

In [11]:
ordered_cols = [
    "clasificacion", "cliente_libre_defendible", "cliente_regulado_probable", "requiere_revision_manual",
    "RBD", "nombre_establecimiento", "cod_comuna", "comuna", "COD_DEPE2", "glosa_dependencia",
    "tipo_dependencia", "matricula_total", "razon_social_cliente", "variantes_razon_social",
    "comuna_cliente", "region_cliente", "anio_min", "anio_max", "n_meses_registrados",
    "retiro_mwh_total", "retiro_mwh_promedio_mensual", "score", "regla_match",
    "alias_establecimiento", "alias_cliente", "comuna_consistente", "tiene_senal_educacional", "posible_educacion_superior",
    "n_candidatos_evaluados", "nota_metodologica",
]

df_resultado = df_cruce[ordered_cols].sort_values(["clasificacion", "score"], ascending=[True, False]).reset_index(drop=True)
df_matches = df_resultado[df_resultado["cliente_libre_defendible"]].copy()
df_revision = df_resultado[df_resultado["requiere_revision_manual"]].copy()
df_regulados = df_resultado[df_resultado["cliente_regulado_probable"]].copy()

path_matches = OUTPUT_DIR / "cruce_educacion_clientes_libres_matches.csv"
path_revision = OUTPUT_DIR / "cruce_educacion_clientes_libres_revision_manual.csv"
path_regulados = OUTPUT_DIR / "cruce_educacion_clientes_libres_regulados_probables.csv"
path_resumen = OUTPUT_DIR / "resumen_educacion_regulados_por_comuna.csv"
path_clasificacion = OUTPUT_DIR / "resumen_educacion_clientes_libres_clasificacion.csv"

df_matches.to_csv(path_matches, index=False, encoding="utf-8-sig")
df_revision.to_csv(path_revision, index=False, encoding="utf-8-sig")
df_regulados.to_csv(path_regulados, index=False, encoding="utf-8-sig")
resumen_comunal.to_csv(path_resumen, index=False, encoding="utf-8-sig")
resumen_clasificacion.to_csv(path_clasificacion, index=False, encoding="utf-8-sig")

print("Archivos exportados:")
for path in [path_matches, path_revision, path_regulados, path_resumen, path_clasificacion]:
    print("-", path)

Archivos exportados:
- c:\Users\Raimundo Claren\Documents\MERLIN_EDM\prototipo_3\data\interim\cruce_educacion_clientes_libres_matches.csv
- c:\Users\Raimundo Claren\Documents\MERLIN_EDM\prototipo_3\data\interim\cruce_educacion_clientes_libres_revision_manual.csv
- c:\Users\Raimundo Claren\Documents\MERLIN_EDM\prototipo_3\data\interim\cruce_educacion_clientes_libres_regulados_probables.csv
- c:\Users\Raimundo Claren\Documents\MERLIN_EDM\prototipo_3\data\interim\resumen_educacion_regulados_por_comuna.csv
- c:\Users\Raimundo Claren\Documents\MERLIN_EDM\prototipo_3\data\interim\resumen_educacion_clientes_libres_clasificacion.csv


## 10. Sintesis metodologica

- La unidad de analisis principal es el establecimiento educacional activo (`ESTADO_ESTAB == 1`) identificado por `RBD`.
- La categoria `publico` agrupa `COD_DEPE2` 1, 4 y 5; la categoria `comercial` agrupa `COD_DEPE2` 2 y 3.
- El indicador principal cuenta como cliente libre solo los matches de alta confianza o probables con senal educacional explicita en la razon social de clientes libres.
- Los casos `revision_manual` se contabilizan como regulados probables en el indicador conservador, hasta que exista validacion externa.
- El cruce es nominal y no usa RUT, RBD en la base electrica, numero de cliente ni identificador de medidor. Por eso `cliente_regulado_probable` debe leerse como una inferencia operacional, no como certificacion tarifaria administrativa.
- Universidades e institutos de educacion superior detectados en clientes libres se marcan aparte, porque no necesariamente corresponden a establecimientos escolares del Directorio Oficial EE.